# back-fn-call-with-recipe-args — faded example 3: Feed the cached out (node.array) into sigmoid_back

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `back-fn-call-with-recipe-args`. Running the beacon reports progress on the `Backprop: back fn call with recipe args` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: back fn call with recipe args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`back-fn-call-with-recipe-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "back-fn-call-with-recipe-args"
DD_SUBTOPIC = "Backprop: back fn call with recipe args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The second positional of the canonical call is `node.array` — the cached forward output — not the input and not the MiniTensor wrapper. `sigmoid_back` uses `out * (1 - out)` for its local derivative, so it relies on receiving the cached `out` in that slot.

## Faded exercise 3

### Faded — supply the correct `out` channel

A node came from `t.sigmoid(x)`. The `sigmoid_back` body and the surrounding call are written, but the **second positional argument** (the cached output channel) is blanked. Fill in the expression that supplies the cached forward `out` so the chain-rule term `out * (1 - out)` is computed against the right tensor.

**Fill in:** the cached forward output passed as the second positional, i.e. node.array

In [ ]:
def sigmoid_back(grad_out, out, x):
    return grad_out * out * (1 - out)

class Recipe:
    def __init__(self, func, args, kwargs):
        self.func, self.args, self.kwargs = func, args, kwargs

class Node:
    def __init__(self, array, recipe):
        self.array, self.recipe = array, recipe

def call_sigmoid_back(node, grad_out):
    cached_out = node.array
    return sigmoid_back(grad_out, cached_out, *node.recipe.args, **node.recipe.kwargs)

t.manual_seed(0)
x = t.randn(6)
out = t.sigmoid(x)
node = Node(out, Recipe(t.sigmoid, (x,), {}))

def _test():
    grad_out = t.ones(6)
    dx = call_sigmoid_back(node, grad_out)
    xr = node.recipe.args[0].clone().requires_grad_(True)
    t.sigmoid(xr).sum().backward()
    assert tuple(dx.shape) == (6,), f'wrong shape {tuple(dx.shape)}'
    assert t.allclose(dx, xr.grad), 'must use cached out (node.array) so out*(1-out) is correct'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def sigmoid_back(grad_out, out, x):
    return grad_out * out * (1 - out)

class Recipe:
    def __init__(self, func, args, kwargs):
        self.func, self.args, self.kwargs = func, args, kwargs

class Node:
    def __init__(self, array, recipe):
        self.array, self.recipe = array, recipe

def call_sigmoid_back(node, grad_out):
    cached_out = node.array
    return sigmoid_back(grad_out, cached_out, *node.recipe.args, **node.recipe.kwargs)

t.manual_seed(0)
x = t.randn(6)
out = t.sigmoid(x)
node = Node(out, Recipe(t.sigmoid, (x,), {}))
```
</details>